# Pincode Matching from Latitude / Longitude

Match each property's coordinates against the official pincode boundary polygons
(`pincode_boundaries/`) and:

1. **Fill the existing `Pincode` column** — rows that already have a pincode keep it;
   rows missing one get the pincode of the polygon their coordinate falls inside.
2. **Add `Possible_Pincodes`** — the **best 3 nearest** pincode zones to the point
   (nearest first), since coordinates carry some jitter and the true pincode may be a neighbour.

Output: `datasets/housedata_updated.csv`

In [1]:
import json
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape
from shapely.strtree import STRtree

df = pd.read_csv('datasets/All_Merged_Updated(in).csv', low_memory=False)
print('Rows:', len(df))
print('Pincode already filled:', df['Pincode'].notna().sum())
print('Pincode missing:', df['Pincode'].isna().sum())

Rows: 93621
Pincode already filled: 24366
Pincode missing: 69255


## Load pincode boundary polygons

In [2]:
polygons, pincodes = [], []
for path in ['pincode_boundaries/telangana.geojson',
             'pincode_boundaries/andhra-pradesh.geojson']:
    gj = json.load(open(path))
    for feat in gj['features']:
        geom = shape(feat['geometry'])
        pc = str(feat['properties'].get('pincode', '')).strip()
        if pc and geom.is_valid:
            polygons.append(geom)
            pincodes.append(pc)

polygons = np.array(polygons, dtype=object)
pincodes = np.array(pincodes)
tree = STRtree(polygons)
print(len(polygons), 'pincode polygons indexed')

1880 pincode polygons indexed


## Match coordinates to pincode zones

For every row with coordinates we compute the distance to nearby pincode polygons
(distance is 0 when the point is inside a polygon), keep the **3 nearest**, and use
the single nearest one to fill missing `Pincode` values.

In [3]:
has_coords = df['Latitude'].notna() & df['Longitude'].notna()
points = shapely.points(df.loc[has_coords, 'Longitude'].to_numpy(),
                        df.loc[has_coords, 'Latitude'].to_numpy())
row_ids = df.index[has_coords].to_numpy()

# candidate zones within ~5 km of each point (covers the coordinate jitter)
RADIUS_DEG = 5.0 / 111.0
buffered = shapely.buffer(points, RADIUS_DEG, quad_segs=4)
pt_idx, poly_idx = tree.query(buffered, predicate='intersects')
dist = shapely.distance(points[pt_idx], polygons[poly_idx])

cand = pd.DataFrame({'row': row_ids[pt_idx],
                     'pincode': pincodes[poly_idx],
                     'dist': dist})
cand = (cand.sort_values(['row', 'dist'])
            .drop_duplicates(['row', 'pincode']))   # one entry per zone per point

# best (nearest) zone per point, and top-3 list nearest-first
best = cand.groupby('row').first()['pincode']
top3 = cand.groupby('row')['pincode'].agg(lambda s: ', '.join(s.head(3)))
print('Points matched to at least one zone:', len(best), 'of', int(has_coords.sum()), 'with coordinates')

Points matched to at least one zone: 77475 of 87795 with coordinates


## Fill `Pincode` and add `Possible_Pincodes`

In [4]:
before = df['Pincode'].notna().sum()

# keep existing pincodes; fill only the missing ones with the nearest matched zone
df['Pincode'] = df['Pincode'].astype('Int64').astype('string')
df['Pincode'] = df['Pincode'].fillna(best.reindex(df.index).astype('string'))

df['Possible_Pincodes'] = top3.reindex(df.index)

after = df['Pincode'].notna().sum()
print(f'Pincode filled: {before} -> {after} rows ({before/len(df)*100:.1f}% -> {after/len(df)*100:.1f}%)')
print('Rows with Possible_Pincodes:', df['Possible_Pincodes'].notna().sum())
df.loc[has_coords, ['Locality', 'Latitude', 'Longitude', 'Pincode', 'Possible_Pincodes']].head(10)

Pincode filled: 24366 -> 79048 rows (26.0% -> 84.4%)
Rows with Possible_Pincodes: 77475


,Locality,Latitude,Longitude,Pincode,Possible_Pincodes
0,Bachupally,17.431084,78.437625,500073,"500045, 500073, 500082"
1,Bachupally,17.431084,78.437625,500073,"500045, 500073, 500082"
2,Madhapur,17.468087,78.372830,500084,"500084, 500081, 500085"
3,Madhapur,17.448584,78.390804,500081,"500081, 500033, 500018"
4,Madhapur,17.448584,78.390804,500081,"500081, 500033, 500018"
5,Madhapur,17.448584,78.390804,500081,"500081, 500033, 500018"
6,Madhapur,17.445049,78.390344,500081,"500081, 500033, 500018"
7,Madhapur,17.448584,78.390804,500081,"500081, 500033, 500018"
8,Madhapur,17.452791,78.398358,500018,"500018, 500081, 500033"
9,Madhapur,17.445049,78.390344,500081,"500081, 500033, 500018"


## Save `housedata_updated.csv`

In [5]:
OUT = 'datasets/housedata_updated.csv'
df.to_csv(OUT, index=False)
print('Saved:', OUT)
print(len(df), 'rows x', len(df.columns), 'columns (added: Possible_Pincodes)')

Saved: datasets/housedata_updated.csv
93621 rows x 31 columns (added: Possible_Pincodes)
